# Complexity-03 — Plus de temps, plus de problèmes

**Navigation** : [Index](README.md) | [Version Python](Complexity-03-HierarchieTemporelle-Python.ipynb) | [02 — Vérifier ou trouver](Complexity-02-P-NP-Reduction-Python.ipynb) | [03b — Hartmanis–Stearns](Complexity-03b-HartmanisStearns-HierarchieTemporelle-Python.ipynb) | [03c — Zoo navigable](Complexity-03c-ComplexityZoo-Navigation-Python.ipynb) | [English →](Complexity-03-HierarchieTemporelle-Python_en.ipynb)

> **Public : Licence.** La démonstration historique et sa simulation multitête sont conservées dans l'approfondissement 03b.

## Ce que ce notebook suppose

**Environnement** : bases de C#, .NET 9 ou ultérieur et noyau .NET Interactive. Le code de ce jumeau utilise uniquement la BCL; les grandes valeurs de budget sont calculées avec `System.Numerics.BigInteger`.

- [01 — Compter des pas](Complexity-01-StepCounting-Python.ipynb) : taille et budget de pas ;
- [02 — Vérifier ou trouver](Complexity-02-P-NP-Reduction-Python.ipynb) : problème de décision et algorithme.

**Objectifs** : distinguer inclusion et séparation stricte de classes, comprendre une diagonale finie et expliquer pourquoi une table n'est pas une preuve de hiérarchie. **Durée estimée** : 35 minutes.

**Le théorème en une phrase.** Le théorème de hiérarchie temporelle (Hartmanis–Stearns, 1965) affirme que, pour des budgets raisonnables, disposer de strictement plus de temps permet de décider strictement plus de problèmes : certaines tâches exigent réellement plus de pas, et aucun algorithme rapide ne peut les simuler. Ce carnet illustre les **mécanismes** de cette preuve — budgets imbriqués, diagonalisation, simulation bornée — sur des tables finies ; il ne prétend pas la démontrer, ce que rappelle chaque lecture.

## 1. Des budgets imbriqués

Si un programme répond en n pas, il répond aussi dans un budget de n² pas. Cela montre une **inclusion** des budgets, pas une séparation des classes de problèmes : deux programmes différents sur un même problème ne prouvent pas qu'aucun programme rapide ne peut le résoudre. Comparons les budgets sur cinq tailles.

L'inclusion est triviale : un programme qui s'arrête en n pas s'arrête aussi avant n² pas. La difficulté est ailleurs — montrer qu'un problème exige *plus* que n² pas demande d'écarter **tous** les programmes rapides, pas seulement celui qu'on a sous la main. C'est ce fossé entre « j'ai un exemple » et « j'ai épuisé les possibilités » que ce carnet creuse.

In [1]:
#nullable enable
using System;
using System.Collections.Generic;
using System.Globalization;
using System.Linq;
using System.Numerics;
Console.WriteLine($"Runtime observé : .NET {Environment.Version}; budgets calculés exactement avec BigInteger.");
var tailles = new[] {2,4,8,16,32};
BigInteger Puissance2(int n) => BigInteger.One << n;
var budgets = tailles.Select(n => (n, Carre: new BigInteger(n)*n, Exponentiel: Puissance2(n))).ToArray();
Console.WriteLine($"{"n",3} | {"n²",12} | {"2ⁿ",14} | n²≤2ⁿ");
foreach(var x in budgets) Console.WriteLine($"{x.n,3} | {x.Carre,12} | {x.Exponentiel,14} | {x.Carre<=x.Exponentiel}");
if(budgets.Length!=5 || budgets.Any(x=>x.Carre>x.Exponentiel) || budgets[^1].Carre!=1024 || budgets[^1].Exponentiel!=4294967296) throw new Exception("Contrôle des cinq budgets exacts échoué");
Console.WriteLine("Contrôle fini : 5 tailles, les inclusions numériques demandées sont vérifiées.");

Runtime observé : .NET 10.0.14; budgets calculés exactement avec BigInteger.


  n |           n² |             2ⁿ | n²≤2ⁿ


  2 |            4 |              4 | True


  4 |           16 |             16 | True


  8 |           64 |            256 | True


 16 |          256 |          65536 | True


 32 |         1024 |     4294967296 | True


Contrôle fini : 5 tailles, les inclusions numériques demandées sont vérifiées.


**Lecture du résultat.** Les coûts s'ordonnent pour les tailles affichées : à n = 32, le budget n² vaut 1 024 pas contre 4 294 967 296 pour 2ⁿ — l'écart est nul à n = 2 et n = 4 (4 contre 4, 16 contre 16) et ne s'ouvre qu'à partir de n = 8 (64 contre 256). Pour conclure qu'une classe possède un problème absent de l'autre, il faudrait écarter **tous** les algorithmes au petit budget. La table numérique ne le fait pas.

## Exercices

Les trois exercices prolongent les démonstrations sur les mêmes objets finis. Ils ne demandent pas de prouver la hiérarchie — hors de portée d'une table — mais de **sentir** ce que chaque table montre et ce qu'elle laisse ouvert, en manipulant budgets, diagonale et borne de simulation.

### Exercice 1 — Inclusion ou séparation ?

Produisez `comparaisonBudgets`, liste de quatre tuples `(n, n², n³)` pour n = 2, 4, 8, 16. Expliquez pourquoi ces nombres ne prouvent pas qu'un problème devient soluble uniquement avec le second budget.

- **Indice :** une table de nombres n'écarte aucun autre algorithme.
- **Etape 1 :** produire les quatre couples.
- **Etape 2 :** distinguer inclusion et séparation.

In [2]:
List<(int n,BigInteger n2,BigInteger n3)> comparaisonBudgets = null; // TODO etudiant : couples (n², n³) pour n=2,4,8,16
if(comparaisonBudgets is { } reponse) { var attendus=new[]{2,4,8,16}.Select(n=>(n,new BigInteger(n)*n,BigInteger.Pow(new BigInteger(n),3))).ToList(); if(reponse.Count!=4 || !reponse.SequenceEqual(attendus) || reponse.Any(x=>x.n2>x.n3)) throw new Exception("Exercice 1 : couples n²/n³ incorrects"); Console.WriteLine($"Exercice 1 vérifié : {string.Join("; ",reponse.Select(x=>$"n={x.n}: {x.n2}/{x.n3}"))}"); } else Console.WriteLine("Exercice à compléter : fournir les quatre couples exacts n²/n³.");

Exercice à compléter : fournir les quatre couples exacts n²/n³.


## 2. L'idée de diagonalisation

Énumérons quelques machines candidates. À la ligne i, la construction D choisit l'opposé de la réponse de la machine numéro i sur l'entrée i. D diffère donc de chaque machine listée sur au moins une entrée. La liste ci-dessous n'est qu'une expérience **finie** : elle ne représente pas toutes les machines possibles.

La diagonalisation inverse le sens de la question : au lieu de chercher un algorithme rapide, on construit un problème D qui contredit chaque machine candidate sur son propre terrain — l'entrée i. Contre une liste finie, cela marche ; contre l'infinité des machines, il faudrait une construction et une borne universelles, ce que la section 4 discute.

In [3]:
Func<int,bool>[] machines = {
    i => i % 2 == 0,
    i => i < 2,
    i => true,
    i => i % 3 == 0
};
var diagonale = Enumerable.Range(0,machines.Length).Select(i=>!machines[i](i)).ToArray();
Console.WriteLine($"{"i",2} | {"Mᵢ(i)",7} | {"D(i)",5} | différents");
for(int i=0;i<machines.Length;i++) Console.WriteLine($"{i,2} | {machines[i](i),7} | {diagonale[i],5} | {diagonale[i]!=machines[i](i)}");
if(machines.Length!=4 || diagonale.Length!=4 || Enumerable.Range(0,4).Any(i=>diagonale[i]==machines[i](i))) throw new Exception("Contrôle de la diagonale finie échoué");
Console.WriteLine("Contrôle fini : quatre machines contredites à leur propre indice.");

 i |   Mᵢ(i) |  D(i) | différents


 0 |    True | False | True


 1 |    True | False | True


 2 |    True | False | True


 3 |    True | False | True


Contrôle fini : quatre machines contredites à leur propre indice.


**Lecture du résultat.** Les quatre lignes affichent `True` dans la colonne « différents » : D(i) vaut l'opposé de M_i(i) sur chacune des entrées listées (0, 1, 2 et 3). Il reste une infinité de machines non listées ; ce tableau ne prouve donc aucune séparation de classes.

### Exercice 2 — Une autre diagonale

Renseignez `nouvelleDiagonale` avec une cinquième fonction `machine` et une liste `rows` de cinq tuples `(i, réponse, diagonale)`. Chaque réponse doit évaluer la machine correspondante sur son propre indice et chaque valeur diagonale doit en être l’opposé. Expliquez pourquoi cette table finie n’est toujours pas une preuve générale.

- **Indice :** évaluer la nouvelle fonction sur son propre indice.
- **Etape 1 :** ajouter une machine.
- **Etape 2 :** recalculer les cinq oppositions.

In [4]:
(Func<int,bool> machine,List<(int i,bool machineResponse,bool diagonal)> rows)? nouvelleDiagonale = null; // TODO etudiant : cinquième machine et cinq oppositions
if(nouvelleDiagonale is { } reponse) { if(reponse.machine is null || reponse.rows is null || reponse.rows.Count!=5) throw new Exception("Exercice 2 : la cinquième machine et les cinq lignes sont requises"); var toutes=machines.Append(reponse.machine).ToArray(); var attendus=Enumerable.Range(0,5).Select(i=>(i,MachineResponse:toutes[i](i),Diagonal:!toutes[i](i))).ToList(); if(!reponse.rows.SequenceEqual(attendus)) throw new Exception("Exercice 2 : indices/réponses/oppositions ne concordent pas"); Console.WriteLine($"Exercice 2 vérifié : 5 machines, 5 contradictions à leur propre indice."); } else Console.WriteLine("Exercice à compléter : fournir une cinquième machine et sa table diagonale à cinq lignes.");

Exercice à compléter : fournir une cinquième machine et sa table diagonale à cinq lignes.


## 3. Le budget de simulation

Une machine qui ne répond pas avant la limite ne peut pas bloquer indéfiniment le constructeur diagonal. Ici les coûts sont **déclarés** par un modèle fini : ce ne sont pas des pas mesurés par un simulateur universel. Une réponse non observée n'est pas un verdict faux.

Dans la preuve réelle, cette limite est gérée par une **horloge universelle** : on simule chaque machine candidate pas à pas et on la stoppe au-delà du budget. Payer à la fois la machine et son simulateur est précisément le surcoût qui fait tenir la hiérarchie — et que le modèle fini ici ne rend pas.

In [5]:
var candidates = new (bool verdict,int pas)[] {(true,1),(false,3),(true,12),(false,40)};
BigInteger Budget(int i) => 2*i+4;
Console.WriteLine($"{"i",2} | {"pas",3} | {"budget",6} | {"observé",8} | {"D(i)",5}");
for(int i=0;i<candidates.Length;i++) { var c=candidates[i]; bool? observe=c.pas<=Budget(i)?c.verdict:null; bool? d=observe.HasValue?!observe.Value:true; Console.WriteLine($"{i,2} | {c.pas,3} | {Budget(i),6} | {observe?.ToString()??"null",8} | {d?.ToString()??"null",5}"); }
if(candidates.Length!=4 || candidates.Select((c,i)=>c.pas<=Budget(i)).SequenceEqual(new[]{true,true,false,false})==false) throw new Exception("Contrôle des quatre coûts déclarés échoué");
Console.WriteLine("Contrôle fini : dépassements à 12>8 et 40>10; null signifie non observé.");

 i | pas | budget |  observé |  D(i)


 0 |   1 |      4 |     True | False


 1 |   3 |      6 |    False |  True


 2 |  12 |      8 |     null |  True


 3 |  40 |     10 |     null |  True


Contrôle fini : dépassements à 12>8 et 40>10; null signifie non observé.


**Lecture du résultat.** `null` signifie « non observé dans le budget », jamais « faux » : aux lignes 2 et 3, les coûts déclarés (12 et 40) dépassent le budget (8 et 10), la simulation s'interrompt et la réponse n'est pas observée. Un dépassement contredit cette limite exacte sur cette entrée. Il ne suffit pas à exclure un temps en O(f(n)), qui autorise des constantes et un seuil asymptotique ; une classe de langages porte en outre sur l’existence d’un algorithme, pas sur cette seule machine. Le tableau ne caractérise aucune classe entière.

## 4. Le théorème, sans raccourci expérimental

Le théorème de hiérarchie temporelle établit, sous des hypothèses techniques, qu'il existe des problèmes décidables avec un budget suffisamment plus grand mais impossibles avec le plus petit. Il faut énumérer toutes les machines admissibles, construire une horloge et payer le coût de simulation universelle. Les tables finies n'accomplissent aucune de ces preuves. [03b — Hartmanis–Stearns](Complexity-03b-HartmanisStearns-HierarchieTemporelle-Python.ipynb) détaille le modèle historique. Ce résultat ne résout pas P contre NP.

Concrètement, la séparation s'obtient en construisant un problème D qui simule chaque machine candidate M_i sur son entrée i, répond l’opposé, et respecte un budget suffisamment plus grand pour payer l’horloge et le surcoût de simulation. La séparation exige les hypothèses du théorème ; une simple augmentation ponctuelle du budget ne la garantit pas. Le prix est connu : énumérer toutes les machines, les doter d'une horloge universelle, et simuler chacune en payant la machine et son simulateur.

### Exercice 3 — Dépassement de budget

Repérez les lignes dont le coût déclaré dépasse `Budget(i)`. Renseignez `budgetEtendu` avec un budget qui couvre les quatre coûts et expliquez pourquoi cela ne construit pas une borne pour toutes les machines.

- **Indice :** comparer les deux colonnes numériques.
- **Etape 1 :** produire un nouveau budget.
- **Etape 2 :** énoncer la limite de ce modèle fini.

In [6]:
int? budgetEtendu = null; // TODO etudiant : nouveau budget couvrant les quatre coûts déclarés
if(budgetEtendu is int b) { if(b<candidates.Max(x=>x.pas) || candidates.Any(x=>x.pas>b)) throw new Exception("Exercice 3 : le nouveau budget ne couvre pas les quatre coûts"); var observes=candidates.Select(x=>x.pas<=b).ToArray(); if(observes.Length!=4 || observes.Any(x=>!x)) throw new Exception("Exercice 3 : tous les coûts doivent être observés"); Console.WriteLine($"Exercice 3 vérifié : budget {b}, quatre coûts couverts [{string.Join(", ",candidates.Select(x=>x.pas))}]."); } else Console.WriteLine("Exercice à compléter : proposer un budget qui couvre chaque coût déclaré.");

Exercice à compléter : proposer un budget qui couvre chaque coût déclaré.


## Exercices — bilan des trois travaux

Les exercices sont répartis dans le parcours, après l’expérience dont ils partent. Le premier vérifie les quatre paires de budgets exactes; le deuxième complète la liste finie avec une cinquième machine et contrôle les cinq oppositions; le troisième demande un budget qui observe chacun des quatre coûts déclarés. Aucun ne démontre une séparation asymptotique : les conclusions restent celles des expériences finies.

| Exercice | Contrôle fourni | Conclusion permise |
|---|---|---|
| 1 — Inclusion ou séparation ? | Quatre couples `(n, n², n³)` aux tailles demandées | Les nombres ne prouvent pas une séparation de problèmes |
| 2 — Une autre diagonale | Cinq indices, réponses et oppositions cohérentes avec les machines | La contradiction porte sur cette liste finie |
| 3 — Dépassement de budget | Un budget unique couvre les quatre coûts `1, 3, 12, 40` | Couvrir ces cas n’établit pas une horloge universelle |

## Conclusion

| Geste | Vérifié ici | Non démontré ici |
|---|---|---|
| Budgets | Inclusion numérique | Séparation stricte de classes |
| Diagonale | Quatre contradictions finies | Toutes les machines admissibles |
| Simulation | Arrêt sur dépassement | Horloge et coût universel bornés |

**À retenir.** Trois gestes, trois portées : l'inclusion des budgets se mesure, la diagonalisation se teste sur une liste finie, la simulation bornée s'observe sur un dépassement — mais aucune table finie n'énumère toutes les machines ni ne construit l'horloge universelle. La hiérarchie temporelle est un théorème sur l'infini des algorithmes ; ce carnet en montre les mécanismes sur le fini, pour que la preuve de 03b ait un terrain.

Le théorème affirme l'existence de problèmes demandant réellement plus de temps, et non seulement de programmes lents. L'approfondissement [03b](Complexity-03b-HartmanisStearns-HierarchieTemporelle-Python.ipynb) travaille le résultat historique.

**Référence** : Hartmanis, J. et Stearns, R. E. (1965), *On the computational complexity of algorithms*, Transactions of the American Mathematical Society 117, 285–306.

## Relativisation : ce qu'une diagonale finie ne peut pas trancher

Les trois tableaux de ce carnet — budgets imbriqués, diagonale finie, simulation bornée — illustrent les **mécanismes** d'une preuve de séparation. Le théorème réel (Hartmanis–Stearns 1965) est inconditionnel : il sépare des classes de problèmes, pas seulement des programmes. Mais la technique qui le porte — la diagonalisation — a une limite connue depuis 1975, et cette limite mérite sa place ici parce qu'elle dit exactement ce que ce type de geste peut et ne peut pas trancher.

**Le théorème des oracles (Baker–Gill–Solovay 1975).** Il existe un oracle récursif A tel que `P^A = NP^A`, et un oracle récursif (sparse) B tel que `P^B ≠ NP^B`. Un oracle est une boîte noire gratuite ajoutée aux deux côtés de l'équation ; le résultat fabriqué par BGS montre que la question « P = NP ? » a les **deux** réponses selon l'oracle. Conséquence immédiate : toute technique de preuve qui *relativise* — qui s'applique tel quel en présence d'un oracle arbitraire — est structurellement neutre sur P vs NP, dans les deux sens. Les auteurs l'écrivent dès leur introduction : la diagonalisation ordinaire est du nombre des techniques que l'on attend relativisantes, et l'on ne voit pas non plus de méthode générale de simulation qui échapperait à l'objection. (Baker, Gill, Solovay, *Relativizations of the P NP question*, SIAM Journal on Computing 4(4), 1975, p. 431 — copie archivée au gisement bibliographique du cluster.)

**Ce que cela change pour la diagonale de ce carnet.** Ajoutez mentalement une colonne « oracle » au tableau de la section 2 : la contradiction finie (machines ≠ au point diagonal) survit inchangée. C'est la marque d'un argument qui relativise — il ne peut donc rien conclure sur P vs NP, seulement sur les objets finis qu'il exhibe. Ce n'est pas un défaut du carnet : la séparation de Hartmanis–Stearns, elle, est démontrée ; la leçon est que le geste diagonal enseigné ici est le bon outil pour les hiérarchies de temps, et le mauvais pour la question P vs NP.

**Une nuance de mesure.** Mehlhorn (rapporté §3 de BGS) : les oracles X avec `P^X = NP^X` forment un ensemble *maigre* — en un sens précis, « la plupart » des oracles séparent. L'intuition catégorielle penche donc séparation, et pourtant la question reste ouverte : la relativisation n'est pas un indice de réponse, c'est la preuve que les méthodes standard n'en fournissent pas.

**Écho contemporain.** Le doute exprimé par Wolfram en 2026 — résoudre P vs NP par une preuve finie depuis les axiomes standard, « the results here make me doubt it » — est la face empirique de la même prudence : là où BGS démontre un théorème *sur les techniques de preuve*, le recensement empirique des machines de Turing petites n'offre qu'une attente. Ces deux propos ont des statuts distincts : résultat démontré sur une méthode de preuve et attente empirique. Le second n’ajoute aucune conclusion au théorème des oracles.

## À retenir

1. **Les budgets sont emboîtés, pas égaux.** À $n = 32$, $n^2$ vaut 1 024 pas et $2^n$ vaut 4 294 967 296 : plus de six ordres de grandeur pour la même taille d'entrée. C'est cet écart qui rend « plus de temps » capable d'acheter quelque chose — et c'est ce qu'il faut prouver, pas supposer.
2. **La diagonale différée se mesure sur quatre machines.** $D(i)$ vaut l'opposé de $M_i(i)$ sur les quatre lignes (`différents = True` partout) : c'est une contradiction **construite**, pas un théorème.
3. **La simulation bornée s'arrête sur dépassement.** À $i = 2$, la simulation coûte 12 pas pour un budget de 8 et rend `null` ; à $i = 3$, 40 pas pour un budget de 10, `null` aussi. `null` signifie « non observé dans le budget », jamais « faux » : confondre les deux ferait croire à une réfutation.
4. **Une table finie ne prouve pas la hiérarchie.** La conclusion du carnet le dit en trois colonnes : inclusion numérique vérifiée, séparation stricte non démontrée, horloge et coût universel bornés absents. C'est [03b](Complexity-03b-HartmanisStearns-HierarchieTemporelle-Python.ipynb) qui porte l'énoncé réel, et la section de relativisation qui borne la méthode.

## Vérifiez votre compréhension

1. À $n = 32$, $n^2$ vaut 1 024 et $2^n$ vaut 4 294 967 296. Quelle propriété cette comparaison doit-elle vérifier pour que « plus de temps » achète réellement quelque chose ?
2. À la ligne $i = 2$, la simulation coûte 12 pas pour un budget de 8 et rend `null`. À quoi correspond, en termes de classes, une machine qui dépasse son budget ?
3. Le tableau de la conclusion distingue « vérifié ici » et « non démontré ici ». Quelle ligne relève de la seconde colonne, et pourquoi ne peut-on pas la démontrer avec quatre machines ?
4. La section de relativisation indique qu'il existe un oracle pour lequel P = NP et un autre pour lequel P ≠ NP. Qu'est-ce que cela dit de la diagonalisation comme méthode ?

<details><summary>Réponses</summary>

1. Pour séparer des classes, il faut un problème décidable dans le grand budget et aucun algorithme le décidant dans le petit budget. Les cinq valeurs numériques ne l’établissent pas. Les hypothèses de constructibilité et l’écart nécessaire pour payer la simulation sont étudiés en [03b](Complexity-03b-HartmanisStearns-HierarchieTemporelle-Python.ipynb).
2. Le dépassement indique seulement que la réponse n’a pas été observée sous la limite exacte de cette expérience. Une exception finie n’exclut pas un temps en O(f(n)) ; elle ne classe pas davantage le langage, qui pourrait disposer d’un autre algorithme. `null` ne signifie donc ni faux, ni hors d’une classe de complexité.
3. La séparation stricte se trouve dans la colonne « non démontré ici ». Quatre machines et cinq tailles ne suffisent pas à exclure tous les algorithmes rapides. Une preuve doit traiter toutes les machines admissibles avec une horloge et un surcoût maîtrisés ; mesurer des exemples ne remplace pas cette quantification.
4. L’existence d’oracles donnant les deux relations interdit une preuve de P = NP ou de P ≠ NP qui resterait valable uniformément pour tout oracle. Cela borne les arguments relativisants ; cela n’interdit pas toute technique faisant intervenir une diagonalisation et ne tranche pas la question sans oracle.

</details>


## Pour aller plus loin

- **Carnet suivant** : [Complexity-04 — Décider sans connaître la suite](Complexity-04-OnlineAlgorithms-Python.ipynb) — l'algorithme en ligne : la même question de coût, mais l'information arrive dans le désordre.
- **Dans la même série** : [Complexity-03b — Hartmanis–Stearns](Complexity-03b-HartmanisStearns-HierarchieTemporelle-Python.ipynb) — le théorème réel et son instrumentation pas à pas ; [Complexity-03c — Naviguer le Zoo exécutable](Complexity-03c-ComplexityZoo-Navigation-Python.ipynb) — les deux mondes d'oracles, exécutés.
- **Carnet voisin d'une autre série** : [CSP-1 — Fundamentals](../Search/Part2-CSP/CSP-1-Fundamentals.ipynb) — l'espace de recherche et son coût mesuré, dans une autre série.
- **Lectures** : Baker, Gill et Solovay (1975), *Relativizations of the P=?NP Question*, SIAM J. Comput. 4-4 — le théorème des oracles qui borne la diagonalisation (copie archivée au gisement partagé, dossier `Bibliographie IA/Automata/`) ; Impagliazzo et Paturi (2001), *On the Complexity of k-SAT* (même dossier).